# Marketing Data Analysis
    a.	Preprocess and clean if necessary.
    b.	Build a model predicting “churn”. 
    c.	Remember to comment your code and give rationales for models, algorithms, and approaches. 


## Import Packages

In [0]:
pip install openpyxl

In [0]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# %matplotlib inline
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

import warnings
warnings.filterwarnings("ignore")

import os
# path=os.environ['USERPROFILE']+r'\OneDrive\BDA2'
# path=os.environ['USERPROFILE']+r'\Documents\BDA2'

## Load data

In [0]:
# import pyodbc
# import urllib
# import sqlalchemy

# '''connect to datahub'''

# params_datahub = urllib.parse.quote_plus("DRIVER={SQL Server Native Client 11.0};"
#                                  "SERVER=localhost\SQLEXPRESS;"
#                                  "DATABASE=datahub;"
#                                  "UID=sa;"
#                                  "PWD=user1")

# engine_datahub = sqlalchemy.create_engine("mssql+pyodbc:///?odbc_connect={}".format(params_datahub))

In [0]:
CATALOG_NAME = "workspace"
SCHEMA_NAME = "default"
VOLUME_NAME = "course_data"

# 文件路径应该位于 Volume 中 'BDA2_Data' 文件夹下
# VOLUME_DATA_FOLDER = "BDA2_Data/" 

# 1. 构造完整的 Volume 路径

user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
path = (
    f"/Workspace/Users/{user}/bda_course/BDA2/data/"
)# df=pd.read_sql_table(r"marketing_churn",engine_datahub)
df=pd.read_excel(path+"marketing_churn.xlsx")
df.head()
#dataframe: df
# a*x1, b*x2...= y (Exited)  model 
# a*x1+b*x2... = probability of Exited

## Exploratory Data Analysis(EDA)



### Check missing values and shape
Normally we need to clean the samples, i,e, impute missing values but in this case the data is pretty clean with no missing values. We also check the shape to make sure it matches the meta data info in the document. 

In [0]:
df.info(), df.shape

### Check Churn ratio
The samples are balanced so we can use "Accuracy" metric to measure the performance of the model

In [0]:
target='Exited'
df[target].value_counts().plot(kind='bar')

### Describe the data

Categorical features/Dimensions 

In [0]:
droplist='Surname'
cat_cols=df.select_dtypes(object).drop(droplist,axis=1).columns.tolist()
cat_cols

In [0]:
df.select_dtypes(object).drop(droplist,axis=1)

In [0]:
df.select_dtypes(object).drop(droplist,axis=1).columns.tolist()

In [0]:
help(df.select_dtypes)

In [0]:
import matplotlib.pyplot as plt
categorical_features = cat_cols
fig, ax = plt.subplots(1, len(categorical_features))

for i, categorical_feature in enumerate(df[categorical_features]):
    df[categorical_feature].value_counts().plot(kind="bar", ax=ax[i],figsize=(20,8),rot=90,fontsize=10).set_title(categorical_feature)


Numeric data

#### Histogram
Histogram groups numeric data into bins, displaying the bins as segmented columns and summarize the distribution of a univariate data set. 

In [0]:
droplist=['RowNumber','CustomerId']
num_cols=df.select_dtypes('number').drop(droplist,axis=1).columns.tolist()
num_cols

In [0]:
fig, ax = plt.subplots(1,2, figsize=(10,8))

sns.distplot(df['CreditScore'], ax=ax[0],bins=50)
sns.distplot(df['Age'],ax=ax[1], bins=150)

In [0]:
# import warnings
# warnings.simplefilter(action='ignore', category=FutureWarning)

fig, ax = plt.subplots(3,3, figsize=(25,15))

for i in range(ax.shape[0]):
    for j in range(ax.shape[1]):
            sns.distplot(df[df[num_cols].columns[i*3+j]], ax=ax[i][j],bins=50)
        
        
# for i in range(9):
#     print(i)
#     sns.distplot(df[df[num_cols].columns[i]], ax=ax[i],bins=50)        

In [0]:
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

fig, ax = plt.subplots(3, 3, figsize=(25, 15))

#plot the features except LOCATION_ID and Risk
for i in range(ax.shape[0]):
    for j in range(ax.shape[1]):
#         sns.boxplot(df[df[num_cols].columns[i*2+j]], ax=ax[i][j],orient='v',showfliers=False)
        sns.boxplot(df[df[num_cols].columns[i*3+j]], ax=ax[i][j],orient='v')

In [0]:
df.describe()
# from the "max" row we can see feature PARA_A, PARA_B,Money_Value,History have some potential outliers. 

### Clip, i.e. assigns values outside boundary to boundary values, the data to deal with outliers. 
 Outliers may distort how we see the data. They contain information too so it's a tradeoff; we lose some info but gain a better big picture of the data.

![image-2.png](attachment:image-2.png)

In [0]:
df[num_cols]=df[num_cols].clip(lower=df[num_cols].quantile(0.05), upper=df[num_cols].quantile(0.95),axis=1)

In [0]:
df[num_cols].quantile(0.95)

In [0]:
df[num_cols]

In [0]:
# here we use quantile 0.01 as lower limit and 0.99 upper.
df[num_cols]=df[num_cols].clip(lower=df[num_cols].quantile(0.01), upper=df[num_cols].quantile(0.99),axis=1)
df.describe()

In [0]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
fig, ax = plt.subplots(3, 3, figsize=(25, 15))

for i in range(ax.shape[0]):
    for j in range(ax.shape[1]):
        sns.distplot(df[df[num_cols].columns[i*3+j]], ax=ax[i][j],bins=50)

### Boxplot the data
Boxplot shows the shape of the distribution, its central value, and its variability

In [0]:
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

fig, ax = plt.subplots(3, 3, figsize=(25, 15))

#plot the features except LOCATION_ID and Risk
for i in range(ax.shape[0]):
    for j in range(ax.shape[1]):
        sns.boxplot(df[df[num_cols].columns[i*3+j]], ax=ax[i][j],orient='v',showfliers=False)

In [0]:
df[num_cols].describe()

#### Explain the Boxplots:
The boxplots tell similar story as the Histograms. None of the distributions seem normal.

### Correlation heatmap
Correlation heatmap allows us to see relations between features/attributes. The higher the absolute coefficient, the stronger the correlation is. 

In [0]:
import seaborn as sns
# df=df.query('`Loan ID`.notnull()', engine='python')
sns.pairplot(df.sample(frac=0.001, replace=True).reset_index(drop=True),plot_kws=dict(marker="+", linewidth=1))

# Numerical vs Numerical

In [0]:

# df['Loan Status encoded']=np.where(df['Loan Status']=='Fully Paid',0,1)
# corrMatrix = df.corr()
# plt.figure(figsize = (12,9))
# ax=(sns.heatmap(corrMatrix, annot=True))
# plt.show()

fig, ax = plt.subplots(figsize=(14,11))

ax=(sns.heatmap(df.corr(), annot=True))

chisquare test
https://www.youtube.com/watch?v=misMgRRV3jQ&ab_channel=MathMeeting

# Categorical vs Categorical

In [0]:
from scipy.stats import chisquare,chi2_contingency

v1=df['Exited']

# v2=df['Geography']
v2=df['Gender']
# g, p, dof, expctd=chi2_contingency(pd.crosstab(v1, v2))
print('Chi Square p value =' , chi2_contingency(pd.crosstab(v1, v2))[1])

display(pd.crosstab(v1, v2))
display(pd.crosstab(v1, v2,normalize='index'))
# sns.heatmap(pd.crosstab(v1, v2), annot=True)
sns.heatmap(pd.crosstab(v1, v2,normalize='index'), annot=True,cmap="Blues")


# Categorical vs Numerical

In [0]:
for cat_col in cat_cols:
    for num_col in num_cols:    
        df.groupby(cat_col)[num_col].mean().plot(kind='bar',title=num_col)
        plt.show()    


# Logestic Regression/Modeling

![image-10.png](attachment:image-10.png)![image-9.png](attachment:image-9.png)![image-8.png](attachment:image-8.png)![image-7.png](attachment:image-7.png)![image.png](attachment:image.png)![image-2.png](attachment:image-2.png)![image-3.png](attachment:image-3.png)![image-4.png](attachment:image-4.png)![image-5.png](attachment:image-5.png)![image-6.png](attachment:image-6.png)

## Data processing and cleaning

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [0]:
# import pyodbc
# import urllib
# import sqlalchemy

# '''connect to datahub'''

# params_datahub = urllib.parse.quote_plus("DRIVER={SQL Server Native Client 11.0};"
#                                  "SERVER=localhost\SQLEXPRESS;"
#                                  "DATABASE=datahub;"
#                                  "UID=sa;"
#                                  "PWD=user1")

# engine_datahub = sqlalchemy.create_engine("mssql+pyodbc:///?odbc_connect={}".format(params_datahub))

In [0]:
CATALOG_NAME = "workspace"
SCHEMA_NAME = "default"
VOLUME_NAME = "course_data"

# 文件路径应该位于 Volume 中 'BDA2_Data' 文件夹下
# VOLUME_DATA_FOLDER = "BDA2_Data/" 

# 1. 构造完整的 Volume 路径

user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
path = (
    f"/Workspace/Users/{user}/bda_course/BDA2/data/"
)# df=pd.read_sql_table(r"marketing_churn",engine_datahub)
df=pd.read_excel(path+"marketing_churn.xlsx")
df.head()

### Numerical Variables

In [0]:

droplist='Surname'
cat_cols=df.select_dtypes(object).drop(droplist,axis=1).columns.tolist()

droplist=['RowNumber','CustomerId']
num_cols=df.select_dtypes('number').drop(droplist,axis=1).columns.tolist()

df[num_cols]=df[num_cols].clip(lower=df[num_cols].quantile(0.01), upper=df[num_cols].quantile(0.99),axis=1)

# df=df.query('`Attrition`.notnull()',engine='python')

In [0]:
#replace mnissing value with median, a better representation of the center of the data if it's not normally ditributed

from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='median')
for col in num_cols:
    df[col]=imputer.fit_transform(df[col].values.reshape(-1, 1))



### Categorical Variables

In [0]:
imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
for col in cat_cols:
    df[col]=imputer.fit_transform(df[col].values.reshape(-1, 1))

In [0]:
#encode the attribute
def one_hot(df, cols):
    """
    @param df pandas DataFrame
    @param cols a list of columns to encode 
    @return a DataFrame with one-hot encoding
    """
    for each in cols:
        dummies = pd.get_dummies(df[each], prefix=each, drop_first=False)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop([each], axis=1)
    return df


# cat_cols.remove('Attrition')
df=one_hot(df,cat_cols)
df.head()

In [0]:
x_list = df.columns.tolist()
x_list = [e for e in x_list if e not in ('Exited','Surname','RowNumber','CustomerId')]
df[x_list]

In [0]:
#For better performance use MinMaxScaler to scale and translates each feature individually such that it is in the given range on the training set, e.g. between zero and one.
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_x =pd.DataFrame(scaler.fit_transform(df[x_list]),columns=x_list)

In [0]:
help(MinMaxScaler)

In [0]:
#double check the data to see if there is any missing values and all categorical attributes have been encoded.
df_x

## Create the function to train the model, test it, and visualize the results

In [0]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(df_x, df['Exited'], test_size = .3)


In [0]:
help(LogisticRegression())

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_score, recall_score
from sklearn.metrics import precision_recall_curve, average_precision_score


                   
#splitting the principal training dataset to subtrain and subtest datasets
x_train, x_test, y_train, y_test = train_test_split(df_x, df['Exited'], test_size = .3)


from sklearn.linear_model import LogisticRegression
logit = LogisticRegression()

logit.fit(x_train, y_train)


predictions = logit.predict(x_test)
probabilities = logit.predict_proba(x_test)
    
print('Algorithm:', type(logit).__name__)
print("\nClassification report:\n", classification_report(y_test, predictions))
print("Accuracy Score:", accuracy_score(y_test, predictions)) 


In [0]:
# len(predictions)

# y_test,predictions

In [0]:
#confusion matrix
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import plotly.offline as py
conf_matrix = confusion_matrix(y_test, predictions)

trace=go.Heatmap(z = conf_matrix,x = ["0", "1"],y = ["0", "1"],showscale = False, colorscale = "Picnic")
fig = make_subplots()
fig.add_trace(trace)
py.iplot(fig)

In [0]:
column_df = pd.DataFrame(x_train.columns.tolist())
coefficients = pd.DataFrame(logit.coef_.ravel())
coef_sumry = (pd.merge(coefficients, column_df, left_index=True, 
                               right_index=True, how="left"))
coef_sumry.columns = ["coefficients", "features"]
coef_sumry = coef_sumry.sort_values(by = "coefficients", ascending=False)
display(coef_sumry)
trace = go.Bar(x = coef_sumry["features"], y = coef_sumry["coefficients"])


fig = make_subplots()
fig.add_trace(trace)
py.iplot(fig)

## Feed the parameters to the function created above
Split the data into train dataset and test dataset and use the Hyper Parameters obtained above to generate a Logistic Regression instance and execute the function.

In [0]:
help(LogisticRegression())

### Interpret the results:
<!-- An Accuracy Score of 0.96 on Test data is a very good score with 1 being perfect 100% correct prediction. 
From the confusion Matrix we know that out of 194 predictions, only 7 mistake. Area under curve(True Positive/ False Positive), 
another model performance metric which often is used for unbalanced samples, is 0.965, also near perfect. 
The Feature Importance chart  suggests that  Money_Values, PARA_B, PARA_A, Score and District_Loss are more powerful predictors for Risk. 
 -->
<!-- Overall we have a very good model that can predict Risk. -->

# Productization of your Insights/Recommendations

In [0]:
# df=pd.read_sql_table(r"marketing_churn",engine_datahub)
# df.head()


In [0]:
df_x

In [0]:
logit

In [0]:
df['probabily'] = logit.predict_proba(df_x)[:,1]
df.sort_values(by='probabily',ascending=False).head(1000)


In [0]:
# import pyodbc
# import urllib
# import sqlalchemy

# '''connect to datahub'''

# params_datahub = urllib.parse.quote_plus("DRIVER={SQL Server Native Client 11.0};"
#                                  "SERVER=localhost\SQLEXPRESS;"
#                                  "DATABASE=datahub;"
#                                  "UID=sa;"
#                                  "PWD=user1")

# engine_datahub = sqlalchemy.create_engine("mssql+pyodbc:///?odbc_connect={}".format(params_datahub))
# # df=df.drop('Purpose_other',axis=1)
# df.to_sql('Marketing_prediction',engine_datahub,if_exists='replace',index=False)